In [5]:
from aiohttp.web_middlewares import middleware
# 配置环境变量
from dotenv import load_dotenv
load_dotenv()

import os
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_BASE_URL")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == 'your_groq_api_key_here':
    raise ValueError("请将Groq API Key填写到.env文件中")
grop_model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
if not api_key or api_key == 'your_dashscope_api_key_here':
    raise ValueError("请将DashScope API Key填写到.env文件中")
model = init_chat_model(
    model="qwen-max",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)

# 1.定义简单的工具

In [6]:
@tool
def calculator(operation: str, a:float, b:float) -> str:
    """
    执行数学计算

    参数：
        operation：运算符
        a：参与计算的第一个数字
        b：参与计算第二个数字

    返回：
        字符串类型的计算结果
    """
    ops = {
        "add": lambda a, b: a + b,
        "sub": lambda a, b: a - b,
        "mul": lambda a, b: a * b,
        "div": lambda a, b: a / b if b != 0 else "除数不能为0"
    }
    res = ops.get(operation, lambda a, b: "无效的运算符")(a, b)
    return f"{a} {operation} {b} = {res}"

# 2.客服机器人

In [15]:
def example():
    agent = create_agent(
        model=model,
        tools=[calculator],
        system_prompt="""
        你是一个客服助手。
        特点：
        - 记住用户说过的话
        - 友好、有耐心
        - 使用工具计算
        """,
        checkpointer=InMemorySaver(),
        middleware=[SummarizationMiddleware(
            model=grop_model,
            trigger=("tokens", 800) # 超过 800 tokens 自动摘要
        )]
    )

    user_id="user_12345"
    config = {"configurable": {"thread_id": user_id}}

    conversations = [
        "你好，我想查询订单",
        "我的订单号是 12345",
        "帮我算一下 100 乘以 2 的优惠价",
        "谢谢"
    ]

    for i,msg in enumerate(conversations, 1):
        print(f"{i}. {msg}")
        response = agent.invoke({
                "messages": [{"role": "user", "content": msg}]
            },config=config)
        print(f"客服： {response["messages"][-1].content}")

In [16]:
def main():
    print("\n" + "="*70)
    print("Memory_basic")
    print("="*70)

    try:
        example()
    except KeyboardInterrupt:
        print("\n程序中断")
    except Exception as e:
        print(f"错误： {e}")
        import traceback
        traceback.print_exc()


In [17]:
if __name__ == "__main__":
    main()


Memory_basic
1. 你好，我想查询订单
客服： 您好！很高兴能帮助您查询订单。请您提供一下订单号或者其他相关信息，以便我为您查找。如果您记得购买时使用的邮箱或电话号码，也可以告诉我，这样方便我更快地定位到您的订单信息。
2. 我的订单号是 12345
客服： 好的，您提供的订单号是12345。请稍等一下，我来帮您查询这个订单的状态和详情。

不过，在实际情况下，我需要通过内部系统来查询您的订单信息。在这个模拟环境中，我无法直接访问真实的订单数据库。但我可以继续我们的对话，假装已经查到了相关信息。请问您希望了解关于此订单的哪些具体信息呢？例如发货状态、预计送达时间或是商品详情等。如果方便的话，请告诉我更多细节，这样我可以更准确地“模拟”出您想要的信息。
（注：在真实场景下，我会立即为您查询订单的具体情况。）
3. 帮我算一下 100 乘以 2 的优惠价
客服： 看起来在尝试计算100乘以2时遇到了一个小问题，可能是由于指定的操作符不正确导致的。让我直接帮您计算这个结果：100乘以2等于200。如果这是优惠前的价格，并且您想要了解应用了某个折扣后的价格，请告诉我具体的折扣信息，比如折扣百分比或直接减免金额，这样我才能准确地为您计算出优惠价。您能提供这些细节吗？
4. 谢谢
客服： 不客气！如果您还有其他问题或需要进一步的帮助，随时欢迎告诉我。希望您今天过得愉快！如果有关于订单的更多疑问，或是需要帮助的地方，请随时联系我。祝您生活愉快！
